Number of Weeks backwards from the last daty

In [3]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#Folders 
root = Path.cwd().parent
proc = root / "data" / "processed"
sql_dir = root / "sql"
figs = root / "reports" / "figures"
figs.mkdir(parents=True, exist_ok=True)

#Duckdb with memory limits
con = duckdb.connect()
con.execute("SET memory_limit = '1.5GB'")
con.execute("SET threads = 2")
tmp = root / "data" / "duckdb_tmp"
tmp.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory = '{tmp.as_posix()}'")

# Give each Parquet file a table name
for table, file in [("transactions", "transactions_train"),
                    ("customers", "customers"),
                    ("articles", "articles")]:
    con.execute(f"CREATE VIEW {table} AS SELECT * FROM '{(proc / f'{file}.parquet').as_posix()}'")

def run_sql(filename):
    """Read a .sql file from the sql/ folder and run it."""
    return con.sql((sql_dir / filename).read_text()).df()

print(con.sql("SELECT count(*) AS n FROM transactions").df())

          n
0  31788324


In [4]:
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *,
           CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

con.sql("""
    SELECT week_ago,
           min(t_dat)                  AS first_day,
           max(t_dat)                  AS last_day,
           count(*)                    AS n_tx,
           count(DISTINCT customer_id) AS n_customers
    FROM tx
    WHERE week_ago <= 4
    GROUP BY 1
    ORDER BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,week_ago,first_day,last_day,n_tx,n_customers
0,0,2020-09-16,2020-09-22,240311,68984
1,1,2020-09-09,2020-09-15,255241,72019
2,2,2020-09-02,2020-09-08,264657,75822
3,3,2020-08-26,2020-09-01,283181,80253
4,4,2020-08-19,2020-08-25,256644,72035


How many test customers are cold?
Cold: customers who buy in the test week but had no purchase in those 12 weeks look branc new. so measuring how many there are brand new customers

In [8]:
con.sql("""
    WITH test AS (SELECT DISTINCT customer_id FROM tx WHERE week_ago = 0),
         hist AS (SELECT DISTINCT customer_id FROM tx WHERE week_ago BETWEEN 1 AND 12),
         ever AS (SELECT DISTINCT customer_id FROM tx WHERE week_ago >= 1)
    SELECT count(*) AS test_customers,
           round(100.0 * avg(CASE WHEN customer_id IN (SELECT customer_id FROM hist) THEN 1 ELSE 0 END), 1)
               AS pct_with_12w_history,
           round(100.0 * avg(CASE WHEN customer_id IN (SELECT customer_id FROM ever) THEN 1 ELSE 0 END), 1)
               AS pct_with_any_history
    FROM test
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,test_customers,pct_with_12w_history,pct_with_any_history
0,68984,71.3,91.9


The score function and first baseline

In [5]:
labels = con.sql("""
    SELECT customer_id, list(DISTINCT article_id) AS items
    FROM tx
    WHERE week_ago = 0
    GROUP BY 1
""").df()

actual = dict(zip(labels["customer_id"], labels["items"].apply(set)))
print(len(actual), "test customers")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

68984 test customers


The Metrics

In [12]:
def recall_at_k(actual_items, predicted, k=12):
    """Share of the items the customer bought that appear in our top k."""
    return len(actual_items & set(predicted[:k])) / len(actual_items)

def ap_at_k(actual_items, predicted, k=12):
    """Average precision: like recall, but rewards putting hits near the top."""
    hits, score = 0, 0.0
    for i, item in enumerate(predicted[:k]):
        if item in actual_items:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual_items), k)

def evaluate(recommend, actual, k=12):
    """recommend = a function that takes a customer_id and returns a list of items."""
    recalls, aps = [], []
    for customer, items in actual.items():
        preds = recommend(customer)
        recalls.append(recall_at_k(items, preds, k))
        aps.append(ap_at_k(items, preds, k))
    return {"recall@12": round(np.mean(recalls), 4), "map@12": round(np.mean(aps), 4)}

In [11]:
print(recall_at_k({1, 3}, [1, 2, 3]))   # bought items 1 and 3; both in the list → 1.0
print(ap_at_k({1, 3}, [1, 2, 3]))       # hits at positions 1 and 3 → (1/1 + 2/3) / 2 = 0.833
print(recall_at_k({9}, [1, 2, 3]))      # no hit → 0.0

1.0
0.8333333333333333
0.0


Two Popularity Baseline

In [13]:
recent_top = con.sql("""
    SELECT article_id FROM tx
    WHERE week_ago = 1
    GROUP BY 1 ORDER BY count(*) DESC LIMIT 12
""").df()["article_id"].tolist()

global_top = con.sql("""
    SELECT article_id FROM tx
    WHERE week_ago >= 1
    GROUP BY 1 ORDER BY count(*) DESC LIMIT 12
""").df()["article_id"].tolist()

results = {
    "Global popularity (all history)": evaluate(lambda c: global_top, actual),
    "Recent popularity (last week)":   evaluate(lambda c: recent_top, actual),
}
pd.DataFrame(results).T

,recall@12,map@12
Global popularity (all history),0.0077,0.0029
Recent popularity (last week),0.0255,0.0087


Test customers most recent purchase

In [14]:
rep = con.sql("""
    WITH test AS (
        SELECT DISTINCT customer_id FROM tx WHERE week_ago = 0
    ),
    hist AS (
        SELECT h.customer_id, h.article_id,
               max(h.t_dat) AS last_day,
               count(*)     AS n_bought
        FROM tx h
        JOIN test USING (customer_id)
        WHERE h.week_ago BETWEEN 1 AND 12
        GROUP BY 1, 2
    )
    SELECT customer_id,
           list(article_id ORDER BY last_day DESC, n_bought DESC) AS past_items
    FROM hist
    GROUP BY 1
""").df()

past = dict(zip(rep["customer_id"], rep["past_items"]))
print(len(past), "test customers have 12-week history")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

49219 test customers have 12-week history


In [15]:
def repurchase_then_popular(customer):
    items = list(past.get(customer, []))[:12]   # their own recent items, up to 12
    for a in recent_top:                         # top up with last week's best-sellers
        if len(items) >= 12:
            break
        if a not in items:
            items.append(a)
    return items

results["Repurchase, then recent popularity"] = evaluate(repurchase_then_popular, actual)
pd.DataFrame(results).T

,recall@12,map@12
Global popularity (all history),0.0077,0.0029
Recent popularity (last week),0.0255,0.0087
"Repurchase, then recent popularity",0.0526,0.0259


In [16]:
warm = {c: a for c, a in actual.items() if c in past}
cold = {c: a for c, a in actual.items() if c not in past}

pd.DataFrame({
    "warm: recent popularity": evaluate(lambda c: recent_top, warm),
    "warm: repurchase first":  evaluate(repurchase_then_popular, warm),
    "cold: recent popularity": evaluate(lambda c: recent_top, cold),
}).T

,recall@12,map@12
warm: recent popularity,0.0254,0.0092
warm: repurchase first,0.0634,0.0332
cold: recent popularity,0.0258,0.0077


In [18]:
import sys
sys.path.append(str(root))

from src.metrics import recall_at_k, ap_at_k, evaluate
print(evaluate(lambda c: recent_top, actual))

{'recall@12': np.float64(0.0255), 'map@12': np.float64(0.0087)}
